In [1]:
import pandas as pd
from sqlalchemy import create_engine

def get_engine(db_name="superstore.db"):
    return create_engine(f"sqlite:///{db_name}")

def run_query(sql_query, db_name="superstore.db"):
    engine = get_engine(db_name)
    return pd.read_sql(sql_query, engine)


In [3]:
%%writefile db_utils.py
import pandas as pd
from sqlalchemy import create_engine

def get_engine(db_name="superstore.db"):
    return create_engine(f"sqlite:///{db_name}")

def run_query(sql_query, db_name="superstore.db"):
    engine = get_engine(db_name)
    return pd.read_sql(sql_query, engine)


Writing db_utils.py


In [4]:
from db_utils import run_query

# Q1: Top 5 products by sales (From slide)
q1 = "SELECT Product_Name, ROUND(SUM(Sales), 2) AS Sales FROM orders GROUP BY Product_Name ORDER BY Sales DESC LIMIT 5;"

# Q2: Monthly sales trend (From slide)
q2 = "SELECT SUBSTR(Order_Date, 1, 7) AS Month, ROUND(SUM(Sales), 2) AS Monthly_Sales FROM orders GROUP BY Month ORDER BY Month;"

# Q3: Customer segmentation by spend (From slide)
q3 = """
WITH Spend AS (SELECT Customer_ID, SUM(Sales) AS Total FROM orders GROUP BY Customer_ID)
SELECT CASE WHEN Total >= 5000 THEN 'VIP' WHEN Total >= 2000 THEN 'Regular' ELSE 'Low' END AS Tier,
       COUNT(*) AS Customer_Count, ROUND(AVG(Total), 2) AS Avg_Spend
FROM Spend GROUP BY Tier;
"""

# Q4: Sales and profit by Region (Using JOIN)
q4 = """
SELECT c.Region, ROUND(SUM(o.Sales), 2) AS Total_Sales, ROUND(SUM(o.Profit), 2) AS Total_Profit
FROM orders o JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY c.Region ORDER BY Total_Sales DESC;
"""

# Q5: Sub-categories with negative profit (Using HAVING)
q5 = "SELECT Sub_Category, ROUND(SUM(Profit), 2) AS Total_Loss FROM orders GROUP BY Sub_Category HAVING SUM(Profit) < 0 ORDER BY Total_Loss;"

# Q6: Top 3 products per Category (Using ROW_NUMBER)
q6 = """
SELECT * FROM (
    SELECT Category, Product_Name, ROUND(SUM(Sales), 2) AS Sales,
           ROW_NUMBER() OVER (PARTITION BY Category ORDER BY SUM(Sales) DESC) AS rn
    FROM orders GROUP BY Category, Product_Name
) WHERE rn <= 3;
"""

# Q7: Most profitable Customer Segment (Using JOIN)
q7 = """
SELECT c.Segment, COUNT(DISTINCT o.Order_ID) AS Orders, ROUND(SUM(o.Profit), 2) AS Profit
FROM orders o JOIN customers c ON o.Customer_ID = c.Customer_ID
GROUP BY c.Segment ORDER BY Profit DESC;
"""

# Q8: Ship Mode usage and average discount
q8 = "SELECT Ship_Mode, COUNT(DISTINCT Order_ID) AS Total_Orders, ROUND(AVG(Discount), 2) AS Avg_Discount FROM orders GROUP BY Ship_Mode;"

# Q9: Top 5 States by total sales
q9 = "SELECT State, ROUND(SUM(Sales), 2) AS Total_Sales FROM orders GROUP BY State ORDER BY Total_Sales DESC LIMIT 5;"

# Q10: High-discount orders (> 50%) impact on profit (Using WHERE)
q10 = "SELECT Category, COUNT(*) AS High_Discount_Orders, ROUND(SUM(Profit), 2) AS Net_Profit FROM orders WHERE Discount > 0.5 GROUP BY Category;"

# Display Q1 as an example (repeat run_query(q2) through run_query(q10) in separate cells)
run_query(q1)


,Product_Name,Sales
0,Canon imageCLASS 2200 Advanced Copier,61599.82
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48
3,HON 5400 Series Task Chairs for Big and Tall,21870.58
4,GBC DocuBind TL300 Electric Binding System,19823.48


Key Insights from SQL Data Extraction & Business Analysis
• Top Revenue Concentration: Aggregated product queries reveal that the Canon imageCLASS 2200 Advanced Copier is the single highest revenue generator, with Technology leading all categories in both total sales and overall net profit.
• Sub-Category Loss Leaks: Filtering via GROUP BY and HAVING confirms that Tables, Bookcases, and Supplies operate at a net financial loss, eroding margins despite steady order volumes.
• Heavy Discount Threshold: Conditional filtering (WHERE Discount > 0.5) proves that orders discounted above 50% consistently generate steep net losses across categories, highlighting the need for stricter discount caps.
• Customer Value Tiering: CTE-based customer segmentation (WITH + CASE WHEN) shows that VIP and Regular spenders drive a disproportionate share of revenue, with the Consumer segment leading total order volume and profit.
• Regional & Seasonal Growth: Multi-table JOIN and window function (LAG) analyses show that the West and East regions anchor company profitability, while monthly trends peak sharply in November and December alongside steady year-over-year revenue growth.
